# EDA Base Template — DuckDB Workflow

**Template ID:** EDA-BASE  
**Workflow layer:** `raw` (after source → raw ingest)  
**Primary mode:** Notebook + SQL via DuckDB Python API

## Purpose

Notebook-first exploratory data analysis template for DuckDB pipelines. Use it to profile a table in the **raw**, **staging**, or **curated** layer before cleaning, validation, or transformation.

Follows the layered workflow convention:

```text
source → raw → staging → curated → output

```

## How to use

1. Copy this notebook into your project (or run it from this repo).
2. Update the **Project configuration** cell with your paths, table name, and column lists.
3. Run all cells top to bottom.
4. Record **Key findings** and **Next actions** at the end.

**Target users:** analysts, data engineers, GIS users, SQL users, Python users.

## 2. Project configuration

Edit the variables below for your dataset. Leave lists empty (`[]`) or `GEOMETRY_COLUMN = None` when a section does not apply.

In [ ]:
from pathlib import Path

# --- paths ---
PROJECT_ROOT = Path("..").resolve()  # repo root when notebook lives in notebooks/
DB_PATH = PROJECT_ROOT / "work.duckdb"
RAW_DATA_PATH = PROJECT_ROOT / "data" / "source" / "your_file.csv"  # source file or folder

# --- table ---
RAW_TABLE_NAME = "raw.raw_your_table"  # schema.table in DuckDB

# --- source format (used only when registering data in this notebook) ---
# Options: csv | parquet | json | shapefile | geojson | geoparquet | gdb
SOURCE_FORMAT = "csv"

# --- columns to profile ---
KEY_COLUMNS = ["id"]  # business / natural keys for duplicate checks
DATE_COLUMNS = []  # e.g. ["order_date", "created_at"]
NUMERIC_COLUMNS = []  # e.g. ["amount", "population"]
CATEGORICAL_COLUMNS = []  # e.g. ["country", "status"]
GEOMETRY_COLUMN = None  # e.g. "geom" for spatial layers; None to skip spatial checks

# --- optional date-range bounds (for date range checks) ---
MIN_EXPECTED_DATE = "1900-01-01"
MAX_EXPECTED_DATE = "2100-12-31"

# --- preview settings ---
PREVIEW_LIMIT = 20
CATEGORY_TOP_N = 20

## 3. Imports

In [ ]:
import duckdb
from IPython.display import display

## 4. Connect to DuckDB

Opens (or creates) a persistent database and ensures workflow schemas exist.

In [ ]:
con = duckdb.connect(str(DB_PATH))

for schema in ("raw", "staging", "curated"):
    con.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}";')

print(f"Connected to: {DB_PATH}")
display(con.sql("SELECT version() AS duckdb_version").df())

## 5. Load extensions

Load extensions required for your source format. Spatial formats need the `spatial` extension.

In [ ]:
EXTENSIONS = ["httpfs", "json"]

if SOURCE_FORMAT in {"shapefile", "geojson", "geoparquet", "gdb"} or GEOMETRY_COLUMN:
    EXTENSIONS.append("spatial")

for ext in dict.fromkeys(EXTENSIONS):  # preserve order, remove duplicates
    con.execute(f"INSTALL {ext};")
    con.execute(f"LOAD {ext};")
    print(f"Loaded extension: {ext}")

## 6. Register source data

Materialize the source file into the `raw` layer. **Skip this cell** if the table already exists from a prior ingest notebook or SQL script.

Set `REGISTER_SOURCE = True` to run registration; leave `False` when profiling an existing table.

In [ ]:
REGISTER_SOURCE = False  # set True to load RAW_DATA_PATH into RAW_TABLE_NAME

source_path = RAW_DATA_PATH.resolve().as_posix()
table_sql = RAW_TABLE_NAME  # expects schema.table

if REGISTER_SOURCE:
    if SOURCE_FORMAT == "csv":
        sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT * FROM read_csv_auto('{source_path}');
        """
    elif SOURCE_FORMAT == "parquet":
        sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT * FROM read_parquet('{source_path}');
        """
    elif SOURCE_FORMAT == "json":
        sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT * FROM read_json_auto('{source_path}');
        """
    elif SOURCE_FORMAT in {"shapefile", "geojson", "geoparquet", "gdb"}:
        sql = f"""
        CREATE OR REPLACE TABLE {table_sql} AS
        SELECT * FROM ST_Read('{source_path}');
        """
    else:
        raise ValueError(f"Unsupported SOURCE_FORMAT: {SOURCE_FORMAT}")

    con.execute(sql)
    print(f"Registered {RAW_DATA_PATH.name} → {RAW_TABLE_NAME}")
else:
    print(f"Skipping registration. Profiling existing table: {RAW_TABLE_NAME}")

## 7. Preview rows

Quick visual scan of sample records.

In [ ]:
preview_sql = f"""
SELECT *
FROM {RAW_TABLE_NAME}
LIMIT {PREVIEW_LIMIT};
"""

display(con.sql(preview_sql).df())

## 8. Inspect schema

Column names, types, and nullability.

In [ ]:
schema_sql = f"DESCRIBE {RAW_TABLE_NAME};"
display(con.sql(schema_sql).df())

## 8b. Consolidated table summary (DuckDB SQL)

Single-call profile via DuckDB SQL: dtypes, nulls, cardinality, top values, numeric stats, and outlier counts. Keeps data in DuckDB — Python only orchestrates SQL.

See [Table summary](../docs/04_eda/table_summary.md) for SQL building blocks. Sections 10–14 below remain for column-specific SQL deep-dives.

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "python"))

from eda_helpers import get_table_summary

exclude_cols = [GEOMETRY_COLUMN] if GEOMETRY_COLUMN else []
table_summary = get_table_summary(
    con,
    RAW_TABLE_NAME,
    print_summary=False,
    properties_as_columns=False,
    top_n=10,
    exclude=[c for c in exclude_cols if c],
)
display(table_summary)

## 9. Row counts

Total row count for the table.

In [ ]:
row_count_sql = f"""
SELECT COUNT(*) AS row_count
FROM {RAW_TABLE_NAME};
"""

display(con.sql(row_count_sql).df())

## 10. Null profiling

Null counts and percentages per column. When `KEY_COLUMNS`, `DATE_COLUMNS`, `NUMERIC_COLUMNS`, and `CATEGORICAL_COLUMNS` are all empty, profiles every column from `DESCRIBE`.

In [ ]:
null_columns = sorted(
    set(KEY_COLUMNS + DATE_COLUMNS + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS)
)

if not null_columns:
    null_columns = con.sql(f"DESCRIBE {RAW_TABLE_NAME}").df()["column_name"].tolist()

if null_columns:
    union_blocks = "\n  UNION ALL\n  ".join(
        f"SELECT '{col}' AS column_name, COUNT(*) - COUNT(\"{col}\") AS null_count FROM base"
        for col in null_columns
    )
    null_profile_sql = f"""
    WITH base AS (
      SELECT * FROM {RAW_TABLE_NAME}
    ),
    metrics AS (
      {union_blocks}
    )
    SELECT
      column_name,
      null_count,
      (SELECT COUNT(*) FROM base) AS total_rows,
      ROUND(100.0 * null_count / (SELECT COUNT(*) FROM base), 2) AS null_pct
    FROM metrics
    ORDER BY null_count DESC;
    """
    display(con.sql(null_profile_sql).df())
else:
    print("No columns available for null profiling.")

## 11. Distinct profiling

Cardinality (distinct count) per column — useful for spotting high-cardinality keys vs low-cardinality categories.

In [ ]:
distinct_columns = sorted(
    set(KEY_COLUMNS + DATE_COLUMNS + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS)
)

if not distinct_columns:
    distinct_columns = con.sql(f"DESCRIBE {RAW_TABLE_NAME}").df()["column_name"].tolist()

if distinct_columns:
    union_blocks = "\n  UNION ALL\n  ".join(
        f"SELECT '{col}' AS column_name, COUNT(DISTINCT \"{col}\") AS distinct_count FROM base"
        for col in distinct_columns
    )
    distinct_profile_sql = f"""
    WITH base AS (
      SELECT * FROM {RAW_TABLE_NAME}
    )
    {union_blocks}
    ORDER BY distinct_count DESC;
    """
    display(con.sql(distinct_profile_sql).df())
else:
    print("No columns available for distinct profiling.")

## 12. Duplicate checks

Find duplicate values for business key columns. Configure `KEY_COLUMNS` with one or more columns.

In [ ]:
if KEY_COLUMNS:
    key_select = ", ".join(f'"{col}"' for col in KEY_COLUMNS)
    duplicate_sql = f"""
    SELECT
      {key_select},
      COUNT(*) AS duplicate_count
    FROM {RAW_TABLE_NAME}
    GROUP BY {key_select}
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC
    LIMIT 100;
    """
    result = con.sql(duplicate_sql).df()
    if result.empty:
        print(f"No duplicate key combinations found for: {KEY_COLUMNS}")
    else:
        display(result)
else:
    print("Set KEY_COLUMNS to run duplicate checks.")

## 13. Numeric summaries

Min, max, average, and sample standard deviation for numeric columns.

In [ ]:
if NUMERIC_COLUMNS:
    agg_exprs = ",\n  ".join(
        f"""COUNT(\"{col}\") AS \"{col}_non_null\",
  MIN(\"{col}\") AS \"{col}_min\",
  MAX(\"{col}\") AS \"{col}_max\",
  ROUND(AVG(\"{col}\"), 4) AS \"{col}_avg\",
  ROUND(STDDEV_SAMP(\"{col}\"), 4) AS \"{col}_stddev\""""
        for col in NUMERIC_COLUMNS
    )
    numeric_sql = f"""
    SELECT
      {agg_exprs}
    FROM {RAW_TABLE_NAME};
    """
    display(con.sql(numeric_sql).df())
else:
    print("Set NUMERIC_COLUMNS to run numeric summaries.")

## 14. Categorical frequency

Top values with counts and share of total rows for low-cardinality columns.

In [ ]:
if CATEGORICAL_COLUMNS:
    for column in CATEGORICAL_COLUMNS:
        freq_sql = f"""
        WITH base AS (
          SELECT "{column}" AS category_value
          FROM {RAW_TABLE_NAME}
        ),
        totals AS (
          SELECT COUNT(*) AS total_rows FROM base
        )
        SELECT
          '{column}' AS column_name,
          category_value,
          COUNT(*) AS value_count,
          ROUND(100.0 * COUNT(*) / (SELECT total_rows FROM totals), 2) AS pct_of_rows
        FROM base
        GROUP BY category_value
        ORDER BY value_count DESC
        LIMIT {CATEGORY_TOP_N};
        """
        print(f"--- {column} ---")
        display(con.sql(freq_sql).df())
else:
    print("Set CATEGORICAL_COLUMNS to run categorical frequency tables.")

## 15. Date range checks

Earliest, latest, and out-of-range counts for date or timestamp columns. Adjust `MIN_EXPECTED_DATE` and `MAX_EXPECTED_DATE` in the configuration cell.

In [ ]:
if DATE_COLUMNS:
    for column in DATE_COLUMNS:
        date_sql = f"""
        SELECT
          '{column}' AS column_name,
          COUNT(\"{column}\") AS non_null_count,
          MIN(\"{column}\") AS min_date,
          MAX(\"{column}\") AS max_date,
          COUNT(*) FILTER (
            WHERE \"{column}\" < DATE '{MIN_EXPECTED_DATE}'
               OR \"{column}\" > DATE '{MAX_EXPECTED_DATE}'
          ) AS out_of_range_count
        FROM {RAW_TABLE_NAME};
        """
        display(con.sql(date_sql).df())
else:
    print("Set DATE_COLUMNS to run date range checks.")

## 16. Optional spatial checks

Run when `GEOMETRY_COLUMN` is set. Covers geometry types, spatial extent, null/empty geometries, and invalid geometry counts.

Requires the `spatial` extension (loaded above).

In [ ]:
if GEOMETRY_COLUMN:
    geom = f'"{GEOMETRY_COLUMN}"'

    geom_type_sql = f"""
    SELECT
      ST_GeometryType({geom}) AS geometry_type,
      COUNT(*) AS feature_count
    FROM {RAW_TABLE_NAME}
    WHERE {geom} IS NOT NULL
    GROUP BY ST_GeometryType({geom})
    ORDER BY feature_count DESC;
    """
    print("--- Geometry type counts ---")
    display(con.sql(geom_type_sql).df())

    extent_sql = f"""
    SELECT
      COUNT(*) AS feature_count,
      ST_XMin(ST_Extent({geom})) AS min_x,
      ST_YMin(ST_Extent({geom})) AS min_y,
      ST_XMax(ST_Extent({geom})) AS max_x,
      ST_YMax(ST_Extent({geom})) AS max_y,
      ST_AsText(ST_Extent({geom})) AS extent_wkt
    FROM {RAW_TABLE_NAME}
    WHERE {geom} IS NOT NULL;
    """
    print("--- Spatial extent ---")
    display(con.sql(extent_sql).df())

    null_geom_sql = f"""
    SELECT
      COUNT(*) FILTER (WHERE {geom} IS NULL) AS null_geometry_count,
      COUNT(*) FILTER (WHERE {geom} IS NOT NULL AND ST_IsEmpty({geom})) AS empty_geometry_count,
      COUNT(*) AS total_rows,
      ROUND(
        100.0 * COUNT(*) FILTER (WHERE {geom} IS NULL) / COUNT(*),
        2
      ) AS null_geometry_pct
    FROM {RAW_TABLE_NAME};
    """
    print("--- Null / empty geometry ---")
    display(con.sql(null_geom_sql).df())

    invalid_geom_sql = f"""
    SELECT
      COUNT(*) FILTER (WHERE {geom} IS NOT NULL AND NOT ST_IsValid({geom})) AS invalid_geometry_count
    FROM {RAW_TABLE_NAME};
    """
    print("--- Invalid geometry count ---")
    display(con.sql(invalid_geom_sql).df())
else:
    print("Set GEOMETRY_COLUMN (e.g. 'geom') to run spatial checks.")

## 17. Key findings

Document what you learned during EDA. Replace the bullets below with your observations.

- **Row volume:** _e.g. 12,450 rows — matches source file row count_
- **Schema issues:** _e.g. `amount` stored as VARCHAR; `order_date` has mixed formats_
- **Nulls:** _e.g. 3% null `customer_id`; all key columns populated_
- **Duplicates:** _e.g. 14 duplicate `order_id` values found_
- **Value ranges:** _e.g. dates span 2018–2025; `amount` max looks like an outlier_
- **Spatial (if applicable):** _e.g. all POLYGON; 2 invalid geometries; CRS appears to be EPSG:4326_

## 18. Next actions

Map findings to the next workflow step. Check the boxes that apply to your project.

- [ ] **Cleaning** — cast types, parse dates, trim text (`docs/06_cleaning/`)
- [ ] **Deduplication** — resolve duplicate keys before staging (`sql/cleaning/deduplication.sql`)
- [ ] **Staging build** — promote `raw` → `staging` with standardized columns (`docs/07_transformation/`)
- [ ] **Validation** — row counts, referential integrity, domain checks (`docs/09_validation/`)
- [ ] **Spatial repair** — fix invalid geometries before spatial joins (`docs/06_cleaning/spatial_geometry_cleaning.md`)
- [ ] **Curated layer** — build analysis-ready tables (`docs/07_transformation/`)
- [ ] **Export** — deliver Parquet, CSV, or GeoParquet (`docs/10_export/`)

---

_Close the DuckDB connection when finished, or leave it open for the next notebook in the session._

In [ ]:
# con.close()
# print("Connection closed.")